# Error correction codes

In [ ]:
import numpy as np

## Init

In [9]:
class ECC:
    HemmingDisguise = {
        3: (7, 4),
        4: (15, 11),
        5: (31, 26),
        6: (63, 57),
        7: (127, 120)
    }
    
    def __init__(self, alphabet: list) -> None:
        alpSize = len(alphabet)
        k = 3
        while (2 ** ECC.HemmingDisguise[k][1] < alpSize):
            k += 1
        m, n = ECC.HemmingDisguise[k]
        self.__k__ = k
        alpDict = {}
        alpDictRev = {}
        for i in range(1, alpSize):
            letter = alphabet[i]
            bits = bin(i).ljust(m, "0")
            alpDict[letter] = bits
            alpDictRev[bits] = letter
        self.__m__, self.__n__ = m, n
        self.__alphabetDictionary__ = alpDict
        self.__alphabetDictionaryReversed__ = alpDictRev
        

    def Crypt(self, word: str) -> str:
        charsInBits = [self.__alphabetDictionary__[c] for c in word]
        
        for c in charsInBits:
            buffer = str(c)
            res = str(c)
            for i in range(self.__k__):
                s = 0
                step = 2 ** i
                r = range(i, self.__m__, step)
                for j in r:
                    if buffer[j] == '1':
                        s += 1
                res = res[:step - 1] + '1' if (s & 1 == 1) else '0' + res[step - 1:]

        return res


    def RemoveKsFromSequence(self, sequence: str) -> str:
        res = ''
        m = self.__m__
        l = len(sequence)
        for i in range(0, l, m):
            c = str(sequence[i : i + m])
            # removing previous k-s
            for j in range(self.__k__):
                step = 2 ** j
                c = c[:step - 1] + c[step:]
            res += c

        return res
    

    def Decrypt(self, cryptedWord: str, isThereKs: bool = True) -> str:
        buffer = RemoveKsFromSequence(cryptedWord) if isThereKs else str(cryptedWord)
        
        res = ''
        bitsCount = len(buffer)
        step = int(bitsCount // self.__m__)
        for i in range(0, bitsCount, step):
           res += self.__alphabetDictionaryReversed__[buffer[i:i + step]] 

        return res


    def MakeErrors(self, cryptedWord: str, errorsCount: int = 1) -> str:
        res = str(cryptedWord)
        l = len(cryptedWord)

        for _ in range(errorsCount):
            index = np.random.randint(0, l)
            c = cryptedWord[index]
            cryptedWord[index] = '0' if c == '1' else '1'

        return res


    def Check(self, sequence: str) -> list:
        ks = [[] * self.__k__]
        m = self.__m__
        l = len(sequence)

        for i in range(0, l, m):
            c = str(sequence[i : i + m])
            currentKIndex = i // m
            # removing previous k-s
            for j in range(self.__k__):
                step = 2 ** j
                ks[currentKIndex].append(1 if c[step - 1 : step] == '1' else 0)
                c = c[:step - 1] + c[step:]

            # recalculate k-s
            for j in range(self.__k__):
                s = 0
                step = 2 ** j
                r = range(j, m, step)
                for j in r:
                    if c[j] == '1':
                        s += 1
                ks[currentKIndex][j] ^= 1 if (s & 1 == 1) else 0

        return ks


    def FixAndDecrypt(self, sequence: str, errors: list) -> str:
        res = RemoveKsFromSequence(sequence)
        indices = []
        for infoK in errors:
            indices.append(int(str(infoK), 2))

        m = self.__m__
        l = len(sequence)
        res = ''
        for i in range(0, l, m):
            c = str(sequence[i : i + m])
            counter = 0
            index = indices[i // m]
            h = 0
            while (counter < index):
                if (float.is_integer(np.log2(counter))):
                    h += 1
                counter += 1
            value = c[counter + h]
            c[counter + h] = '0' if value == '1' else '1'
            res += c

        return Decrypt(res, False)




## Implementation

In [11]:
alp = [str(i) for i in range(10)]
word = '11308'

# TODO: Tests

ecc = ECC(alp)
encoded = ecc.Crypt(word)
decoded = ecc.Decrypt(word)
print(f'Word: {word}')
print(f'Encoded: {encoded}')
print(f'Decoded: {decoded}')

KeyboardInterrupt: 